Install and Import

In [1]:
import subprocess
subprocess.run(["pip", "install", "groq", "-q"])

import json
import joblib
import pandas as pd
import numpy as np
from groq import Groq

Setup Groq Client

In [2]:
# Paste your Groq API key here
GROQ_API_KEY = "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"

client = Groq(api_key=GROQ_API_KEY)
print("Groq client ready")

Groq client ready


Load Your Trained Model

In [3]:
best_model = joblib.load('../models/best_model.pkl')
print("Prediction model loaded")

Prediction model loaded


Predict Occupancy for All Parks

In [4]:
def predict_all_parks(hour, day_encoded, num_parks=18, prev_occupancy=50.0):
    results = []
    for park_id in range(num_parks):
        sample = pd.DataFrame([{
            'Hour': hour,
            'DayOfWeek_encoded': day_encoded,
            'ParkID_encoded': park_id,
            'OccupancyRate_lag1': prev_occupancy
        }])
        pred = best_model.predict(sample)[0]
        pred = max(0, min(100, pred))
        results.append({
            'ParkID': f"Park_{park_id}",
            'PredictedOccupancy': round(pred, 1)
        })

    return pd.DataFrame(results).sort_values('PredictedOccupancy').reset_index(drop=True)

Build the Prompt

In [5]:
def build_prompt(hour, day_name, predictions_df):

    park_lines = ""

    for _, row in predictions_df.head(5).iterrows():

        occ = row['PredictedOccupancy']

        if occ < 40:
            status = "Low traffic"
        elif occ < 70:
            status = "Moderate traffic"
        else:
            status = "High traffic"

        park_lines += (
            f"- {row['ParkID']}: {occ:.1f}% occupied ({status})\n"
        )

    prompt = f"""
You are Vemo, a friendly and helpful parking assistant.

Your job is to help drivers quickly find the best parking option while saving time, reducing stress, and avoiding traffic.

Current Situation:
- Arrival Time: {hour}:00
- Day: {day_name}

Predicted parking occupancy:

{park_lines}

Please provide:

1. 🚗 Best parking recommendation
   - Recommend the most suitable parking area.
   - Explain the reason in simple words.

2. ⏱ Time saving estimate
   - Compare it with the busiest parking option.
   - Give an approximate number of minutes saved.

3. 🌱 Environmental impact
   - Estimate CO₂ savings in grams.
   - Assume 1.2 g CO₂ saved for every 1% occupancy difference.

4. 💡 Friendly tip
   - Give one short parking or travel tip relevant to this time and day.

Writing Style:
- Be warm, friendly, and encouraging.
- Use simple everyday English.
- Keep the response concise (around 100–150 words).
- Avoid technical terms and complex explanations.
- Sound like a helpful assistant speaking to a driver.

End the response with:
"Safe travels! 🚗💚 – Vemo"
"""

    return prompt

Call Groq LLaMA Model

In [9]:
def get_ai_recommendation(hour, day_encoded, day_name):
    # Get predictions
    predictions = predict_all_parks(hour, day_encoded)

    # Build prompt
    prompt = build_prompt(hour, day_name, predictions)

    # Call Groq LLaMA
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
        {
            "role": "system",
            "content": """
You are Vemo, an intelligent and eco-friendly parking assistant.

Your goals are:
- Help users find the best parking option.
- Reduce parking search time.
- Encourage environmentally friendly travel decisions.
- Give clear, practical recommendations.

Always:
- Use simple and friendly language.
- Keep responses concise and easy to understand.
- Explain recommendations using the provided parking data.
- Focus on helping the user make a quick decision.
- End responses with: 'Safe travels! 🚗💚 – Vemo'
"""
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
        temperature=0.7,
        max_tokens=400
    )

    recommendation = response.choices[0].message.content
    return predictions, recommendation

Test It

In [10]:
# Test: Wednesday 8AM
hour         = 8
day_encoded  = 2
day_name     = "Wednesday"

predictions, recommendation = get_ai_recommendation(hour, day_encoded, day_name)

print("=" * 50)
print("GREENPARK AI RECOMMENDATION")
print("=" * 50)
print(recommendation)
print()
print("Raw Predictions (Top 5):")
print(predictions.head(5).to_string(index=False))

GREENPARK AI RECOMMENDATION
You're almost at your destination! 🚗

**1. Best parking recommendation:**
I recommend Park_5 (20.6% occupied). It's the least crowded option, making it a quick and stress-free choice. You'll likely find a spot right away, and you'll avoid the hassle of circling around other areas.

**2. Time saving estimate:**
By choosing Park_5, you'll save around 7 minutes compared to the busiest option, Park_0 (30.4% occupied). That's a nice chunk of time you can use for a coffee break or some last-minute preparations.

**3. Environmental impact:**
By parking in Park_5, you'll save approximately 25.4 grams of CO₂ compared to Park_0. Every little bit counts, and making eco-friendly choices adds up!

**4. Friendly tip:**
As it's a Wednesday morning, traffic is relatively light. Consider using public transport or carpooling to reduce congestion and make your journey even smoother.

Safe travels! 🚗💚 – Vemo

Raw Predictions (Top 5):
ParkID  PredictedOccupancy
Park_5           

Test Different Times

In [11]:
test_scenarios = [
    (8,  0, "Monday"),
    (12, 4, "Friday"),
    (17, 5, "Saturday"),
]

for hour, day_enc, day_name in test_scenarios:
    print(f"\n{'='*50}")
    print(f"Scenario: {day_name} at {hour}:00")
    print('='*50)
    _, rec = get_ai_recommendation(hour, day_enc, day_name)
    print(rec)


Scenario: Monday at 8:00
Hello there! I'm Vemo, your friendly parking assistant. I've got the parking situation covered for you.

🚗 Best parking recommendation: Park_0 is your best bet! It's only 11.5% occupied, making it the quietest parking option. You'll save yourself the stress of navigating through crowded parking lots and avoid traffic congestion.

⏱ Time saving estimate: By choosing Park_0, you'll save around 6-8 minutes compared to the busiest option. That's a whole extra cup of coffee to enjoy!

🌱 Environmental impact: Choosing the less crowded parking lot will save around 13.8 grams of CO₂ equivalent to a 6-mile drive on electric mode!

💡 Friendly tip: Since it's a Monday, consider carpooling or using public transport to reduce congestion and emissions. It's a great way to start the week!

Safe travels! 🚗💚 – Vemo

Scenario: Friday at 12:00
Hello there, driver! I'm happy to help you find the best parking spot.

**1. Best parking recommendation**
Based on the data, I recommend

Memory Assistant + AI Summary

In [14]:
import spacy

spacy_nlp = spacy.load('../models/spacy_ner_model')

def ai_parking_memory(user_note):

    # Extract entities using SpaCy NER
    doc = spacy_nlp(user_note)

    extracted = {
        ent.label_: ent.text
        for ent in doc.ents
    }

    # Build context-aware prompt
    prompt = f"""
A driver has saved a parking note.

Original Note:
"{user_note}"

Extracted Parking Details:
- Zone: {extracted.get('ZONE', 'Not specified')}
- Floor: {extracted.get('FLOOR', 'Not specified')}
- Landmark: {extracted.get('LANDMARK', 'Not specified')}

Your task:
1. Confirm where the vehicle is parked.
2. Help the driver remember the location.
3. Mention the landmark, zone, and floor if available.
4. Give a simple tip for finding the vehicle later.

Writing Style:
- Friendly and reassuring.
- Use simple everyday English.
- Keep the response short (2–4 sentences).
- Do not sound robotic.
- Speak directly to the driver.

End with:
"See you when you get back! 🚗💚 – Vemo"
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": """
You are Vemo, a friendly parking memory assistant.

Your purpose is to help drivers remember exactly where they parked their vehicles.

Always:
- Use a warm and helpful tone.
- Make the location easy to remember.
- Use the extracted parking information provided.
- Keep responses concise and practical.
- End with:
'See you when you get back! 🚗💚 – Vemo'
"""
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.5,
        max_tokens=150
    )

    summary = response.choices[0].message.content

    return extracted, summary

 Test Memory Assistant

In [15]:
test_notes = [
    "I parked near the blue pillar in Zone A on Level 2",
    "Left my car beside the elevator on Level 3",
    "Near the library entrance close to Zone B"
]

for note in test_notes:
    print("=" * 50)
    print(f"Note     : {note}")
    extracted, summary = ai_parking_memory(note)
    print(f"Extracted: {extracted}")
    print(f"AI Says  : {summary}")
    print()

Note     : I parked near the blue pillar in Zone A on Level 2
Extracted: {'LANDMARK': 'blue pillar', 'ZONE': 'A'}
AI Says  : Don't worry, I've got you covered. You parked your vehicle near the blue pillar in Zone A. To make it even easier, think of the blue pillar as a big blue friend who's waiting for you on Level 2. Just look for that blue pillar and you'll find your car in no time!

See you when you get back! 🚗💚 – Vemo

Note     : Left my car beside the elevator on Level 3
Extracted: {'LANDMARK': 'elevator'}
AI Says  : Don't worry, I've got you covered. You parked your car beside the elevator on Level 3. That's all you need to remember - the elevator on Level 3. Just head back to that spot and you'll find your car waiting for you.

See you when you get back! 🚗💚 – Vemo

Note     : Near the library entrance close to Zone B
Extracted: {'LANDMARK': 'library entrance', 'ZONE': 'B'}
AI Says  : Hi there, don't worry, I've got you covered. Your vehicle is parked near the library entrance, c

Save Everything for the App

In [16]:
# Save config for Streamlit app
config = {
    "groq_model"      : "llama3-8b-8192",
    "num_parks"       : 18,
    "spacy_model_path": "../models/spacy_ner_model",
    "best_model_path" : "../models/best_model.pkl"
}

with open('../models/app_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Config saved to models/app_config.json")
print("Phase 5 complete — ready for Streamlit app")

Config saved to models/app_config.json
Phase 5 complete — ready for Streamlit app
